# Preprocessing y Split Temporal — Provincial

Este notebook ejecuta el pipeline de preprocessing completo siguiendo el protocolo
definido en `planificacion_modelos_provincial.md`. Todos los pasos estan en el
orden correcto para evitar leakage temporal.

El resultado son los arrays numpy listos para entrenar cualquier modelo:
`X_train`, `X_val`, `X_test`, `C_train`, `C_val`, `C_test` y sus variantes de shares.

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import pickle

project_root = Path.cwd().parent  # provincias/
sys.path.append(str(project_root / "src"))

from preprocessing import (
    load_x, load_c, temporal_split, verify_split,
    detect_anomalous_days, remove_anomalous_days,
    fit_x_scaler, transform_x, compute_daily_shares,
    fit_c_scaler, transform_c,
)

PROCESSED_DIR = project_root / "data" / "processed"
SPLIT_DIR     = project_root / "data" / "splits"

print(f"project_root:  {project_root}")
print(f"PROCESSED_DIR: {PROCESSED_DIR}")
print(f"Existe:        {PROCESSED_DIR.exists()}")

project_root:  c:\Users\pazbi\OneDrive\Escritorio\ml\tp_final_ml\provincias
PROCESSED_DIR: c:\Users\pazbi\OneDrive\Escritorio\ml\tp_final_ml\provincias\data\processed
Existe:        True


## 1. Carga de datos

Cargamos `x_perfiles_diarios.csv` y `c_condiciones_diarias.csv` y verificamos
que las fechas esten alineadas entre los dos archivos.

In [3]:
x_df = load_x(PROCESSED_DIR)
c_df = load_c(PROCESSED_DIR)

print(f'x shape: {x_df.shape}')
print(f'c shape: {c_df.shape}')
print(f'x rango: {x_df["fecha"].min().date()} -> {x_df["fecha"].max().date()}')
print(f'c rango: {c_df["fecha"].min().date()} -> {c_df["fecha"].max().date()}')
print(f'Fechas en x pero no en c: {len(set(x_df["fecha"]) - set(c_df["fecha"]))}')
print(f'Fechas en c pero no en x: {len(set(c_df["fecha"]) - set(x_df["fecha"]))}')

x shape: (2311, 529)
c shape: (2311, 356)
x rango: 2019-01-01 -> 2026-04-30
c rango: 2019-01-01 -> 2026-04-30
Fechas en x pero no en c: 0
Fechas en c pero no en x: 0


## 2. Split temporal

Split cronologico estricto. No hay mezcla de fechas entre conjuntos.
El split se hace antes de cualquier transformacion.

In [4]:
x_train, x_val, x_test = temporal_split(x_df)
c_train, c_val, c_test = temporal_split(c_df)

summary = verify_split(x_train, x_val, x_test)
summary

{'train_size': 1460,
 'val_size': 366,
 'test_size': 485,
 'train_years': [2019, 2021, 2022, 2023],
 'val_years': [2024],
 'test_years': [2025, 2026],
 'train_range': (Timestamp('2019-01-01 00:00:00'),
  Timestamp('2023-12-31 00:00:00')),
 'val_range': (Timestamp('2024-01-01 00:00:00'),
  Timestamp('2024-12-31 00:00:00')),
 'test_range': (Timestamp('2025-01-01 00:00:00'),
  Timestamp('2026-04-30 00:00:00')),
 'overlap_train_val': 0,
 'overlap_train_test': 0,
 'overlap_val_test': 0,
 'leakage_detected': False}

In [5]:
if summary['leakage_detected']:
    raise ValueError('Leakage detectado. Revisar las fechas del split.')
print('Split verificado: sin overlap entre conjuntos.')
print(f'  Train: {summary["train_size"]} dias - {summary["train_years"]}')
print(f'  Val:   {summary["val_size"]} dias  - {summary["val_years"]}')
print(f'  Test:  {summary["test_size"]} dias  - {summary["test_years"]}')

Split verificado: sin overlap entre conjuntos.
  Train: 1460 dias - [2019, 2021, 2022, 2023]
  Val:   366 dias  - [2024]
  Test:  485 dias  - [2025, 2026]


## 3. Eliminacion de dias anomalos

Los dias con nulos o ceros simultaneos en varias provincias corresponden
a cortes de luz reales. No se imputan — imputarlos falsificaria los datos.
Se eliminan del dataset y se guardan aparte como benchmark para anomaly detection.

Dias identificados:
- `2023-12-17`: nulos en Catamarca y Jujuy.
- `2023-03-01`: nulo en San Luis.
- `2019-06-16`: ceros simultaneos en varias provincias pequenas.

In [6]:
# Detectar dias anomalos antes del split para tener el panorama completo
anomalies_detected = detect_anomalous_days(x_df)
print(f'Dias anomalos detectados en el dataset completo: {len(anomalies_detected)}')
anomalies_detected

Dias anomalos detectados en el dataset completo: 3


,fecha,reason
0,2023-03-01,null values — possible outage
1,2023-12-17,null values — possible outage
2,2019-06-16,simultaneous zeros — possible outage


In [7]:
# Eliminar dias anomalos de cada split
# Los dias eliminados se guardan para usarlos como benchmark de anomaly detection
x_train, c_train, anomaly_train = remove_anomalous_days(x_train, c_train)
x_val,   c_val,   anomaly_val   = remove_anomalous_days(x_val,   c_val)
x_test,  c_test,  anomaly_test  = remove_anomalous_days(x_test,  c_test)

import pandas as pd
anomaly_log = pd.concat([anomaly_train, anomaly_val, anomaly_test], ignore_index=True)
print(f'Total de dias eliminados: {len(anomaly_log)}')
print(f'  Train: {len(anomaly_train)}, Val: {len(anomaly_val)}, Test: {len(anomaly_test)}')
anomaly_log

Total de dias eliminados: 3
  Train: 3, Val: 0, Test: 0


,fecha,reason
0,2023-03-01 00:00:00,null values — possible outage
1,2023-12-17 00:00:00,null values — possible outage
2,2019-06-16 00:00:00,simultaneous zeros — possible outage


In [8]:
# Verificar que no quedan nulos en ningun split
demand_cols = [c for c in x_train.columns if c != 'fecha']
assert x_train[demand_cols].isnull().sum().sum() == 0, 'Quedan nulos en train'
assert x_val[demand_cols].isnull().sum().sum()   == 0, 'Quedan nulos en val'
assert x_test[demand_cols].isnull().sum().sum()  == 0, 'Quedan nulos en test'
print('Sin valores nulos en ningun split.')
print(f'Dias restantes — train: {len(x_train)}, val: {len(x_val)}, test: {len(x_test)}')

Sin valores nulos en ningun split.
Dias restantes — train: 1457, val: 366, test: 485


In [9]:
demand_cols = [c for c in x_train.columns if c != 'fecha']
assert x_train[demand_cols].isnull().sum().sum() == 0, 'Quedan nulos en train'
assert x_val[demand_cols].isnull().sum().sum()   == 0, 'Quedan nulos en val'
assert x_test[demand_cols].isnull().sum().sum()  == 0, 'Quedan nulos en test'
print('Sin valores nulos en ningun split.')

Sin valores nulos en ningun split.


## 4. Normalizacion de x

Ajustamos el StandardScaler exclusivamente sobre train.
Lo aplicamos a val y test sin reajustar.

Con 528 dimensiones y escalas muy distintas entre provincias
(Buenos Aires supera 10.000 MWh/hora, Santa Cruz raramente supera 200 MWh/hora),
la normalizacion es necesaria para que el modelo no ignore las provincias pequenas.

In [10]:
x_scaler = fit_x_scaler(x_train, method='standard')

X_train = transform_x(x_train, x_scaler)
X_val   = transform_x(x_val,   x_scaler)
X_test  = transform_x(x_test,  x_scaler)

print(f'X_train: {X_train.shape}, media={X_train.mean():.4f}, std={X_train.std():.4f}')
print(f'X_val:   {X_val.shape},   media={X_val.mean():.4f},   std={X_val.std():.4f}')
print(f'X_test:  {X_test.shape},  media={X_test.mean():.4f},  std={X_test.std():.4f}')

X_train: (1457, 528), media=0.0000, std=1.0000
X_val:   (366, 528),   media=0.3081,   std=1.1119
X_test:  (485, 528),  media=0.4511,  std=1.1791


In [11]:
assert abs(X_train.mean()) < 0.01,       'Media de X_train no es ~0'
assert abs(X_train.std() - 1.0) < 0.01, 'Std de X_train no es ~1'
print('Scaler verificado: ajustado correctamente sobre train.')

Scaler verificado: ajustado correctamente sobre train.


## 5. Shares provinciales

Los shares son la variante normalizada de x donde cada provincia-hora
se divide por la demanda nacional de esa hora. No requieren fitting.
Son la base del experimento de Bloque 3.

In [12]:
S_train = compute_daily_shares(x_train)
S_val   = compute_daily_shares(x_val)
S_test  = compute_daily_shares(x_test)

print(f'S_train: {S_train.shape}, min={S_train.min():.4f}, max={S_train.max():.4f}')

s_reshaped = S_train.reshape(-1, 22, 24)
hour_sums = s_reshaped.sum(axis=1)
assert np.allclose(hour_sums, 1.0, atol=1e-6), 'Los shares no suman 1 en alguna hora'
print('Shares verificados: suman 1 por hora en cada dia.')

S_train: (1457, 528), min=0.0001, max=0.5782
Shares verificados: suman 1 por hora en cada dia.


## 6. Normalizacion de c

El vector c combina variables de naturaleza muy distinta.
Ajustamos StandardScaler sobre las variables continuas de train.
Las variables binarias de calendario (es_feriado) se dejan sin escalar.

In [13]:
c_scaler, c_groups = fit_c_scaler(c_train)

print('Grupos de columnas en c:')
for grupo, cols in c_groups.items():
    print(f'  {grupo}: {len(cols)} columnas')
    if len(cols) <= 5:
        print(f'    {cols}')
    else:
        print(f'    {cols[:3]} ... {cols[-2:]}')

Grupos de columnas en c:
  calendar_binary: 1 columnas
    ['es_feriado']
  calendar_ordinal: 2 columnas
    ['mes', 'dia_semana']
  climate: 264 columnas
    ['temp_max_buenos_aires', 'temp_min_buenos_aires', 'temp_media_buenos_aires'] ... ['grado_calor_tucuman', 'grado_frio_tucuman']
  socioeconomic: 66 columnas
    ['poblacion_buenos_aires', 'participacion_industrial_buenos_aires', 'superficie_km2_buenos_aires'] ... ['participacion_industrial_tucuman', 'superficie_km2_tucuman']


In [14]:
C_train = transform_c(c_train, c_scaler, c_groups)
C_val   = transform_c(c_val,   c_scaler, c_groups)
C_test  = transform_c(c_test,  c_scaler, c_groups)

print(f'C_train: {C_train.shape}')
print(f'C_val:   {C_val.shape}')
print(f'C_test:  {C_test.shape}')
print(f'Dimension de c: {C_train.shape[1]} features')

C_train: (1457, 333)
C_val:   (366, 333)
C_test:  (485, 333)
Dimension de c: 333 features


## 7. Verificacion final anti-leakage

Checklist completo antes de guardar los arrays.

In [15]:
checks = {
    'Split sin overlap':         not summary['leakage_detected'],
    'Sin nulos en X_train':      not np.isnan(X_train).any(),
    'Sin nulos en X_val':        not np.isnan(X_val).any(),
    'Sin nulos en X_test':       not np.isnan(X_test).any(),
    'Sin nulos en C_train':      not np.isnan(C_train).any(),
    'Sin nulos en C_val':        not np.isnan(C_val).any(),
    'Sin nulos en C_test':       not np.isnan(C_test).any(),
    'X_train media ~0':          abs(X_train.mean()) < 0.01,
    'X_train std ~1':            abs(X_train.std() - 1.0) < 0.01,
    'Shares suman 1':            np.allclose(S_train.reshape(-1, 22, 24).sum(axis=1), 1.0, atol=1e-6),
}

all_ok = True
for check, result in checks.items():
    status = 'OK' if result else 'FALLO'
    print(f'  [{status}] {check}')
    if not result:
        all_ok = False

print()
if all_ok:
    print('Todos los checks pasaron. Los arrays estan listos para entrenar.')
else:
    print('Hay checks fallidos. Revisar antes de entrenar.')

  [OK] Split sin overlap
  [OK] Sin nulos en X_train
  [OK] Sin nulos en X_val
  [OK] Sin nulos en X_test
  [OK] Sin nulos en C_train
  [OK] Sin nulos en C_val
  [OK] Sin nulos en C_test
  [OK] X_train media ~0
  [OK] X_train std ~1
  [OK] Shares suman 1

Todos los checks pasaron. Los arrays estan listos para entrenar.


## 8. Guardado

Guardamos los arrays numpy y los scalers en `data/splits/`
para que cualquier modelo los pueda cargar sin reprocesar.

In [16]:
SPLIT_DIR.mkdir(parents=True, exist_ok=True)

for name, arr in [
    ('X_train', X_train), ('X_val', X_val), ('X_test', X_test),
    ('C_train', C_train), ('C_val', C_val), ('C_test', C_test),
    ('S_train', S_train), ('S_val', S_val), ('S_test', S_test),
]:
    np.save(SPLIT_DIR / f'{name}.npy', arr)

np.save(SPLIT_DIR / 'dates_train.npy', x_train['fecha'].values)
np.save(SPLIT_DIR / 'dates_val.npy',   x_val['fecha'].values)
np.save(SPLIT_DIR / 'dates_test.npy',  x_test['fecha'].values)

with open(SPLIT_DIR / 'x_scaler.pkl', 'wb') as f:
    pickle.dump(x_scaler, f)
with open(SPLIT_DIR / 'c_scaler.pkl', 'wb') as f:
    pickle.dump(c_scaler, f)
with open(SPLIT_DIR / 'c_groups.pkl', 'wb') as f:
    pickle.dump(c_groups, f)

anomaly_log.to_csv(SPLIT_DIR / 'anomaly_log.csv', index=False)

print(f'Archivos guardados en {SPLIT_DIR}/')
print(f'  x_dim = {X_train.shape[1]}')
print(f'  c_dim = {C_train.shape[1]}')
print(f'  train = {X_train.shape[0]} dias')
print(f'  val   = {X_val.shape[0]} dias')
print(f'  test  = {X_test.shape[0]} dias')

Archivos guardados en c:\Users\pazbi\OneDrive\Escritorio\ml\tp_final_ml\provincias\data\splits/
  x_dim = 528
  c_dim = 333
  train = 1457 dias
  val   = 366 dias
  test  = 485 dias
